# Recommendation Systems Introduction

This notebook provides a hands-on exploration of recommendation systems using the MovieLens 100k dataset. It begins with loading and examining the dataset, followed by interactive sections to better understand its structure. The core focus is on two major classes of recommendation approaches: memory-based methods, which rely on user–user and item–item similarities (e.g., collaborative filtering with cosine similarity), and model-based methods, which use machine learning techniques to capture latent patterns in the data. By walking through both perspectives, the notebook illustrates the intuition, implementation, and comparative strengths of these approaches in generating personalized movie recommendations.

Please note that you will have to select the Tensorflow kernel in order for this notebook to run properly.

## Imports

In [ ]:
# @title ⚙️ Setup: install required packages and download project data
import os

import numpy as np
import pandas as pd
import os
import zipfile
import urllib.request
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import NMF

from IPython.display import HTML

repo_url = "https://github.com/eth-ainit-fs26/coding-exercises.git"
branch = "week2"

if os.path.exists("coding-exercises"):
    !git -C project pull origin {branch}
else:
    !git clone --branch {branch} --single-branch {repo_url}

# Move into the project folder so all imports resolve correctly
if os.path.basename(os.getcwd()) != "recommendation_systems":
    os.chdir("coding-exercises/recommendation_systems")

## Now, let's play a game

In [ ]:
# @title Load Matrix Visualizer {display-mode: "form"}
with open("matrix.html", "r", encoding="utf-8") as f:
    html_content = f.read()

HTML(html_content)

## Warm-Up: The "Toy" Dataset

Before diving into 100k ratings, let's look at how these recommenders work on a tiny, manageable dataset. Imagine we have **3 Users** and **4 Movies**.

* **Movies 1 & 2** are Action movies.
* **Movies 3 & 4** are Romance movies.
* **User A** loves action, hates romance.
* **User B** loves romance, hasn't seen any action.
* **User C** likes a bit of both.

We will try to predict the missing ratings (marked with 0).

In [ ]:
# Create our toy dataset manually
toy_data = pd.DataFrame(
    [
        [5, 5, 1, 0], # User A: Loves action (5s), dislikes romance (1), hasn't seen Movie 4 (0)
        [0, 0, 5, 4], # User B: Hasn't seen action (0s), loves romance (5, 4)
        [3, 0, 2, 5]  # User C: Likes Movie 1 & 4, hasn't seen 2, dislikes 3
    ],
    index=["User A", "User B", "User C"],
    columns=["Movie 1 (Act)", "Movie 2 (Act)", "Movie 3 (Rom)", "Movie 4 (Rom)"]
)

print("Original Toy Ratings Matrix:")
toy_data

### 1. Warm-Up: Matrix Factorization (SVD & NMF)
Just like we will do later with the full dataset, we can "factorize" this matrix. This means breaking it into two smaller matrices: one representing **"User tastes"** and one representing **"Movie features."**
Multiplying them back together gives us our predictions.

In [ ]:
# @title Warm-Up: Toy NMF Factorization {display-mode: "form"}
from sklearn.decomposition import NMF

# We choose n_components=2 because we know there are roughly 2 genres (Action, Romance)
nmf_toy = NMF(n_components=2, init='random', random_state=42, max_iter=500)

# 1. Learn the user tastes and item features
user_tastes = nmf_toy.fit_transform(toy_data)
movie_features = nmf_toy.components_

# 2. Multiply them back together to get predictions
predicted_toy_ratings = np.dot(user_tastes, movie_features)
predicted_toy_df = pd.DataFrame(
    predicted_toy_ratings,
    index=toy_data.index,
    columns=toy_data.columns
)

print("Predicted Ratings (NMF):")
# Notice how it filled in the zeros with reasonable guesses!
predicted_toy_df.round(1)

But, we will also need a way to measure how good our predictions are. A common way is to use the **Frobenius Norm**, which is like a distance measure between the original and reconstructed matrices.

You are asked to implement this function in the following cell, following the instructions provided to you by the comments.

In [ ]:
def norm_frobenius(original_matrix, estimated_matrix):
    # Extract the values as numpy arrays from the DataFrames
    original = original_matrix.values
    reconstructed = estimated_matrix.values

    # Calculate the Frobenius norm
    frobenius_sum = 0

    # Iterate through each element of the matrices to compute the sum of squared differences.
    # Hint: Both matrices have the same shape.
    for i in range(original.shape[0]):
       for j in range(original.shape[1]):
          frobenius_sum += 0 # TODO: fill in this line
          # --- 🎯🎯🎯🎯 ---

    # Lastly, you will need to take the square root of the sum to get the Frobenius norm.
    frobenius_norm = np.sqrt(frobenius_sum)

    return frobenius_norm

In [ ]:
frobenius_error = norm_frobenius(toy_data, predicted_toy_df)
print(f"Frobenius Norm Error between original and predicted ratings: {frobenius_error:.2f}")

### 2. Warm-Up: Neural Network Approach
Neural networks don't usually look at the whole matrix at once. Instead, they look at individual examples of **(User, Movie) -> Rating**.
We have to reshape our toy data into a list of these interactions to "teach" the neural network.

**Note:** To make our tiny example work, we will train on *all* interactions (including 0s). This differs slightly from the main notebook, which has enough data to learn from only the 100,000 *known* ratings.

In [ ]:
# @title Warm-Up: Toy Neural Network Training {display-mode: "form"}
# 1. Reshape data
toy_interactions = toy_data.stack().reset_index()
toy_interactions.columns = ['User', 'Movie', 'Rating']

train_toy = toy_interactions.copy() 
train_toy['User_ID'] = train_toy['User'].astype('category').cat.codes
train_toy['Movie_ID'] = train_toy['Movie'].astype('category').cat.codes
target_ratings = train_toy['Rating'] / 5.0

print("Data reshaped for Neural Network training (now includes 0s):")
display(train_toy) # Will now show 12 rows

latent_dim = 4

# 2. Build the MLP Model
user_in = layers.Input(shape=(1,), name="user")
movie_in = layers.Input(shape=(1,), name="item")
user_emb = layers.Embedding(3, latent_dim, name="user_embedding")(user_in)
movie_emb = layers.Embedding(4, latent_dim, name="movie_embedding")(movie_in)
user_vec = layers.Flatten()(user_emb)
movie_vec = layers.Flatten()(movie_emb)
mlp_concat = layers.Concatenate()([user_vec, movie_vec])
mlp_layer = layers.Dense(16, activation="relu")(mlp_concat)
mlp_layer = layers.Dense(8, activation="relu")(mlp_layer)
predictions = layers.Dense(1, activation="sigmoid")(mlp_layer)

toy_model = keras.Model(inputs=[user_in, movie_in], outputs=predictions)
toy_model.compile(optimizer='adam', loss='mse')

# 3. Train the model (now on all 12 samples)
print("Training model...")
toy_model.fit(
    [train_toy['User_ID'], train_toy['Movie_ID']],
    target_ratings,
    epochs=500,
    verbose=0
)
print("Training complete!")

# 4. Predict and scale back
all_user_ids = np.arange(3)
all_movie_ids = np.arange(4)
predicted_toy_ratings_nn = toy_model.predict(
    [np.repeat(all_user_ids, 4), np.tile(all_movie_ids, 3)]
).reshape(3, 4)

predicted_toy_ratings_nn_df = pd.DataFrame(
    predicted_toy_ratings_nn * 5,
    index=toy_data.index,
    columns=toy_data.columns
)

print("\n--- Predicted Ratings (Neural Network) ---")
display(predicted_toy_ratings_nn_df.round(2))
print("\n--- Original Ratings ---")
display(toy_data)

Once again, we can check how well it learned by predicting the full matrix and comparing with the original using the Frobenius norm.

In [ ]:
frobenius_error = norm_frobenius(toy_data, predicted_toy_ratings_nn_df)
print(f"Frobenius Norm Error between original and predicted ratings: {frobenius_error:.2f}")

Let's jump into a real-world dataset now!

## Real World Dataset - MovieLens 100k

### Downloading Dataset

In [ ]:
# @title Download MovieLens Dataset {display-mode: "form"}
dataset_url = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
dataset_path = "ml-100k.zip"
extract_folder = "ml-100k"

if not os.path.exists(extract_folder):
    if not os.path.exists(dataset_path):
        print("Downloading dataset...")
        urllib.request.urlretrieve(dataset_url, dataset_path)
    with zipfile.ZipFile(dataset_path, "r") as zip_ref:
        zip_ref.extractall(".")

### Loading Dataset

In [ ]:
# @title Load Ratings {display-mode: "form"}
ratings = pd.read_csv(
    os.path.join(extract_folder, "u.data"),
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
    encoding="latin-1"
)

In [ ]:
# @title Load Movies {display-mode: "form"}
import os
movies = pd.read_csv(
    os.path.join(extract_folder, "u.item"),
    sep="|",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "title"],
    encoding="latin-1"
)
movies['movie_id'] = movies['movie_id'].astype(int)

In [ ]:
# @title Build User-Item Matrix {display-mode: "form"}
user_item_matrix = ratings.pivot(index="user_id", columns="item_id", values="rating").fillna(0)

### Dataset Exploration


In [ ]:
# Print the shapes of the datasets to get familiar with their sizes

print("Ratings Dataset Shape:")
# --- 🎯🎯🎯🎯 ---

print("Movies Dataset Shape:")
# --- 🎯🎯🎯🎯 ---

print("User-Item Matrix Shape:")
# --- 🎯🎯🎯🎯 ---


In [ ]:
# Calculate the amount of missing ratings in the user-item matrix.
# For this you will need to extract the values from the user_item_matrix dataframe and check how many of them are zero. Then retrieve the mean of that.
print("Missing Ratings Percentage:")
# --- 🎯🎯🎯🎯 ---


In [ ]:
# Find the most-rated movie ID and its count
# For this you will need to use the value_counts() method paired with max() and idxmax().
most_rated_movie_count = ... # TODO: fill in this line
most_rated_movie_id = ... # TODO: fill in this line
# --- 🎯🎯🎯🎯 ---

print("Most-rated movie ID:", most_rated_movie_id)
print("Most-rated movie count:", most_rated_movie_count)

In [ ]:
# Find the user with the best average rating. Exclude users with less than 20 ratings.
# For this you will need to group by user_id, calculate the mean and count of ratings, filter by count, and then find the max mean.

user_means = ratings.groupby("user_id")["rating"].mean()
user_counts = ratings.groupby("user_id")["rating"].count()
filtered_users = user_counts[user_counts >= 20].index

best_user_id = ... # TODO: fill in this line
best_user_avg_rating = ... # TODO: fill in this line
# --- 🎯🎯🎯🎯 ---

print("User with best average rating (min 20 ratings):", best_user_id)
print("Best average rating:", best_user_avg_rating)

In [ ]:
# Finally, in order to get a more holistic view of the dataset, print the first 5 rows of each dataframe.
print("Ratings Dataset Sample:")
# --- 🎯🎯🎯🎯 ---
print("Movies Dataset Sample:")

# --- 🎯🎯🎯🎯 ---
print("User-Item Matrix Sample:")

# --- 🎯🎯🎯🎯 ---


Now that we have a good understanding of the dataset, let's move on to building recommendation systems using both memory-based and model-based approaches!

## Memory Based

Memory-based recommendation systems, often referred to as neighborhood methods, work directly with the user–item rating matrix to find similarities between users or items. By leveraging similarity measures such as cosine similarity, these methods identify users with comparable preferences or items with similar rating patterns, and then use this information to generate recommendations. They are conceptually simple, intuitive to implement, and provide a clear explanation of why a particular recommendation was made.

### User Cosine Similarity

In [ ]:
# Calculate the user-user cosine similarity matrix from the user-item matrix using the cosine_similarity method.
user_similarity = ... # TODO: fill in this line
# --- 🎯🎯🎯🎯 ---

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("User-User Similarity Matrix Shape:", user_similarity_df.shape)

In [ ]:
user_similarity_df.head()

In [ ]:
# @title User-Based CF Recommender Function {display-mode: "form"}
def recommend_movies(user_id, user_item_matrix, user_similarity, movies, top_n=5):

    # Extract user similarity score with every other user.
    sim_scores = user_similarity[user_id - 1]
    sim_scores = sim_scores.reshape(1, -1)

    # Rescale the ratings giving more weight on similar users
    weighted_ratings = sim_scores.dot(user_item_matrix.values)

    # Produce predictions as an average of the movie rating across users
    sim_sums = np.abs(sim_scores).sum(axis=1)
    pred_ratings = weighted_ratings / (sim_sums + 1e-8)
    preds = pd.Series(pred_ratings.flatten(), index=user_item_matrix.columns)
    # Filter out movies already rated by the user so we do not suggest them again
    watched = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index

    # Extract and return the top rated movies for suggestions that the user has not seen yet
    top_movies = preds.sort_values(ascending=False)
    top_movies = top_movies[~top_movies.index.isin(watched)]
    top_movies = top_movies.head(top_n)

    return movies[movies["movie_id"].isin(top_movies.index)][["movie_id", "title"]].reset_index(drop=True)

In [ ]:
recommend_movies(1, user_item_matrix, user_similarity, movies)

## Model Based

Model-based recommendation systems go beyond direct similarity calculations and instead build predictive models that capture underlying patterns in the data. A common approach is matrix factorization, where the user–item rating matrix is decomposed into latent factors representing hidden characteristics of users and items. These models are generally more scalable and can handle sparse datasets better than memory-based methods, while often achieving higher accuracy. Although they may be less interpretable, model-based techniques form the foundation of many modern large-scale recommendation systems.

### Matrix Factorization with Singular Value Decomposition (SVD)

In [ ]:
# @title SVD Matrix Factorization {display-mode: "form"}
# Get the latent factors of users and items using SVD
svd = TruncatedSVD(n_components=20, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_

# Use them to get a ratings prediction for each pair of a user and a pair
pred_ratings_svd = np.dot(user_factors, item_factors)

In [ ]:
# @title SVD Recommender Function {display-mode: "form"}
def recommend_movies_svd(user_id, pred_ratings, user_item_matrix, movies, top_n=5):
    # Get the predicted rating for the user
    user_pred = pred_ratings[user_id-1]

    # Filter out movies already rated by the user so we do not suggest them again
    watched = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    preds = pd.Series(user_pred, index=user_item_matrix.columns).drop(watched)

    # Extract and return the top rated movies for suggestions
    top_movies = preds.sort_values(ascending=False).head(top_n)
    return movies[movies["movie_id"].isin(top_movies.index)][["movie_id", "title"]]

In [ ]:
recommend_movies_svd(1, pred_ratings_svd, user_item_matrix, movies)

### Matrix Factorization with Non-Negative Matrix Factorization (NMF)

In [ ]:
# Get the latent factors of users and items using the NMF method
nmf = ... # TODO: fill in this line
# --- 🎯🎯🎯🎯 ---

user_factors = ... # TODO: fill in this line
# --- 🎯🎯🎯🎯 ---

item_factors = ... # TODO: fill in this line
# --- 🎯🎯🎯🎯 ---

# Use them to get a ratings prediction for each pair of a user and a pair
pred_ratings = np.dot(user_factors, item_factors)

In [ ]:
# @title NMF Recommender Function {display-mode: "form"}
def recommend_movies_nmf(user_id, pred_ratings, user_item_matrix, movies, top_n=5):
    # Get the predicted rating for the user
    user_pred = pred_ratings[user_id-1]

    # Filter out movies already rated by the user so we do not suggest them again
    watched = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    preds = pd.Series(user_pred, index=user_item_matrix.columns).drop(watched)

    # Extract and return the top rated movies for suggestions
    top_movies = preds.sort_values(ascending=False).head(top_n)
    return movies[movies["movie_id"].isin(top_movies.index)][["movie_id", "title"]]

In [ ]:
recommend_movies_nmf(8, pred_ratings, user_item_matrix, movies)

### Generalized Matrix Factorization (GMF)

Generalized Matrix Factorization extends the basic idea of matrix factorization by making the interaction function between user and item latent factors more flexible. Instead of relying solely on the dot product (as in traditional MF), GMF introduces a neural network layer to learn how user and item embeddings should be combined to predict ratings or preferences. This allows the model to capture more complex, nonlinear relationships between users and items while still retaining the interpretability and efficiency of embedding-based representations. GMF is a core component in modern neural recommendation architectures such as Neural Collaborative Filtering (NCF).

In [ ]:
# @title GMF Data Preparation {display-mode: "form"}
# Setup the data used to train the model
num_users = ratings["user_id"].nunique()
num_items = ratings["item_id"].nunique()

# Scale the data
ratings["rating_norm"] = ratings["rating"] / 5.0

X = [ratings["user_id"].values - 1, ratings["item_id"].values - 1]
y = ratings["rating_norm"].values
latent_dim = 20

In [ ]:
# @title GMF Model Architecture {display-mode: "form"}
# Input Layers
user_input = keras.Input(shape=(1,), name="user")
item_input = keras.Input(shape=(1,), name="item")

# Embedding Layers
user_emb = layers.Embedding(num_users, latent_dim, name="user_embedding")(user_input)
item_emb = layers.Embedding(num_items, latent_dim, name="item_embedding")(item_input)

# Flatten Embeddings
user_vec = layers.Flatten()(user_emb)
item_vec = layers.Flatten()(item_emb)

# Element-wise multiplication between user and item embeddings
interaction = layers.Multiply()([user_vec, item_vec])
output = layers.Dense(1, activation="sigmoid")(interaction)

# Compile the model
model = keras.Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer="adam", loss="mse")

model.summary()

In [ ]:
# @title Train GMF Model {display-mode: "form"}
# Train the model
history = model.fit(
    X, y,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# @title GMF Recommender Function {display-mode: "form"}
def recommend_gmf_keras(user_id, model, movies, top_n=5):

    # Get the model predictions for the user
    user = np.array([user_id-1] * num_items)
    items = np.arange(num_items)
    preds = model.predict([user, items], verbose=0).flatten()

    # Filter out movies already rated by the user so we do not suggest them again
    watched = ratings[ratings["user_id"] == user_id]["item_id"].values
    preds_filtered = {i+1: preds[i] for i in range(num_items) if (i+1) not in watched}

    # Extract and return the top rated movies for suggestions
    top_items = sorted(preds_filtered.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_movie_ids = [movie_id for movie_id, _ in top_items]

    return movies[movies["movie_id"].isin(top_movie_ids)][["movie_id", "title"]]

In [ ]:
recommend_gmf_keras(80, model, movies)

### GMF + MLP

While GMF enriches traditional matrix factorization with learnable interaction functions, combining it with a Multi-Layer Perceptron (MLP) further enhances the model’s ability to capture complex user–item relationships. The GMF component preserves the embedding-based representation and multiplicative interaction, while the MLP introduces nonlinear transformations that can model higher-order correlations. By jointly training both parts and merging their outputs, this hybrid approach—commonly referred to as Neural Collaborative Filtering (NCF)—achieves a balance between interpretability and expressive power, leading to more accurate and flexible recommendations.

In [ ]:
latent_dim = 20
mlp_latent_dim = 40

# Input Layers
user_input = layers.Input(shape=(1,), name="user")
item_input = layers.Input(shape=(1,), name="item")

# Embedding Layers
gmf_user_emb = layers.Embedding(num_users, latent_dim, name="gmf_user_embedding")(user_input)
gmf_item_emb = layers.Embedding(num_items, latent_dim, name="gmf_item_embedding")(item_input)

# Flatten Embeddings
gmf_user_vec = layers.Flatten()(gmf_user_emb)
gmf_item_vec = layers.Flatten()(gmf_item_emb)

# Embeddings Combination
gmf_interaction = layers.Multiply()([gmf_user_vec, gmf_item_vec])

# MLP Embedding Layers
mlp_user_emb = layers.Embedding(num_users, mlp_latent_dim, name="mlp_user_embedding")(user_input)
mlp_item_emb = layers.Embedding(num_items, mlp_latent_dim, name="mlp_item_embedding")(item_input)

# MLP Flatten Embeddings
mlp_user_vec = layers.Flatten()(mlp_user_emb)
mlp_item_vec = layers.Flatten()(mlp_item_emb)

# Define MLP Architecture
mlp_layer = layers.Concatenate()([mlp_user_vec, mlp_item_vec])
# TODO: Feel free to experiment with different architectures using functions for standard MLP
# --- 🎯🎯🎯🎯 ---

# Merge GMF and MLP results
merged = layers.Concatenate()([gmf_interaction, mlp_layer])

# Produce Output
output = layers.Dense(1, activation="sigmoid")(merged)

# Compile the model
model = keras.Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer="adam", loss="mse")

model.summary()

In [ ]:
# @title Train GMF+MLP Model {display-mode: "form"}
# Train the model
history = model.fit(
    X, y,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# @title GMF+MLP Recommender Function {display-mode: "form"}
def recommend_gmf_mlp(user_id, model, movies, top_n=5):

    # Get the model predictions for the user
    user = np.array([user_id-1] * num_items)
    items = np.arange(num_items)
    preds = model.predict([user, items], verbose=0).flatten()


    # Filter out movies already rated by the user so we do not suggest them again
    watched = ratings[ratings["user_id"] == user_id]["item_id"].values
    preds_filtered = {i+1: preds[i] for i in range(num_items) if (i+1) not in watched}

    # Extract and return the top rated movies for suggestions
    top_items = sorted(preds_filtered.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_movie_ids = [movie_id for movie_id, _ in top_items]

    return movies[movies["movie_id"].isin(top_movie_ids)][["movie_id", "title"]]

In [ ]:
recommend_gmf_mlp(100, model, movies)

### GMF + MLP Splitted Embeddings

An extension of the GMF + MLP framework is to split the user and item embeddings into two parts: one dedicated to the GMF branch and the other to the MLP branch. This design prevents the two components from sharing identical representations and allows each branch to specialize—GMF focuses on learning linear interactions, while MLP learns nonlinear patterns. By combining their complementary strengths at the output layer, this approach enhances the overall expressiveness of the model and often leads to better recommendation accuracy compared to using a single shared embedding space.

In [ ]:
latent_dim = 20
mlp_latent_dim = 40

# Define inputs
item_input = layers.Input(shape=(1,), name='item-input')
user_input = layers.Input(shape=(1,), name='user-input')

# GMF Branch Embedding Layers
user_embedding_mf = layers.Embedding(num_users, latent_dim, name='user-embedding-mf')(user_input)
user_vec_mf = layers.Flatten(name='flatten-user-mf')(user_embedding_mf)

movie_embedding_mf = layers.Embedding(num_items, latent_dim, name='movie-embedding-mf')(item_input)
movie_vec_mf = layers.Flatten(name='flatten-movie-mf')(movie_embedding_mf)

# Embeddings Combination
gmf_interaction = layers.Multiply()([user_vec_mf, movie_vec_mf])

# MLP Embeddings
movie_embedding_mlp = layers.Embedding(num_items, mlp_latent_dim, name='movie-embedding-mlp')(item_input)
movie_vec_mlp = layers.Flatten(name='flatten-movie-mlp')(movie_embedding_mlp)

user_embedding_mlp = layers.Embedding(num_users, mlp_latent_dim, name='user-embedding-mlp')(user_input)
user_vec_mlp = layers.Flatten(name='flatten-user-mlp')(user_embedding_mlp)

# Define MLP Architecture
mlp_concat = layers.Concatenate()([user_vec_mlp, movie_vec_mlp])
# TODO: Feel free to experiment with different architectures using functions for standard MLP
# --- 🎯🎯🎯🎯 ---

output = layers.Dot(axes=1, normalize=False, name="pred_mf")([mlp_layer, gmf_interaction])

# Compile the model
model = keras.Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer="adam", loss="mse")

model.summary()

In [ ]:
# @title Train GMF+MLP Split Model {display-mode: "form"}
# Train the model
history = model.fit(
    X, y,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# @title GMF+MLP Split Recommender Function {display-mode: "form"}
def recommend_gmf_mlp_conc(user_id, model, movies, top_n=5):
    # Get the model predictions for the user
    user = np.array([user_id-1] * num_items)
    items = np.arange(num_items)
    preds = model.predict([user, items], verbose=0).flatten()

    # Filter out movies already rated by the user so we do not suggest them again
    watched = ratings[ratings["user_id"] == user_id]["item_id"].values
    preds_filtered = {i+1: preds[i] for i in range(num_items) if (i+1) not in watched}

    # Extract and return the top rated movies for suggestions
    top_items = sorted(preds_filtered.items(), key=lambda x: x[1], reverse=True)[:top_n]
    top_movie_ids = [movie_id for movie_id, _ in top_items]

    return movies[movies["movie_id"].isin(top_movie_ids)][["movie_id", "title"]]

In [ ]:
recommend_gmf_mlp_conc(10, model, movies)